# 珠三角和长三角城市的 gap_to_gdp 的对比分析

-  潘宁，25210220

- 作业思路：
1. 找到珠三角和长三角的城市名单，并生成对应的csv文件
2. 在清洗过的“merged_data.csv”文件中找到对应的长三角和珠三角的城市，分别计算长三角和珠三角城市每年度的gap_to_gdp数据，并绘制折线图。

1. 找到珠三角和长三角的城市名单，并生成对应的csv文件，存放在output文件夹中。

- 提示词相关信息：https://www.doubao.com/thread/w41eff49e9e6ed6da
- 数据源：长三角：依据《长江三角洲区域一体化发展规划纲要》，共 27 市，含上海、江苏 9 市、浙江 9 市、安徽 8 市。粤港澳大湾区：依据《粤港澳大湾区发展规划纲要》，共 11 市，含香港、澳门及广东 9 市

In [1]:
# -*- coding: utf-8 -*-
# 生成中国城市群 CSV 文件，并保存到 output 文件夹
import csv
import os

# 自动创建 output 文件夹（不存在则创建）
os.makedirs("output", exist_ok=True)

# 数据内容
data = [
    ["city", "province", "group"],
    ["上海", "上海市", "长三角"],
    ["南京", "江苏省", "长三角"],
    ["无锡", "江苏省", "长三角"],
    ["常州", "江苏省", "长三角"],
    ["苏州", "江苏省", "长三角"],
    ["南通", "江苏省", "长三角"],
    ["盐城", "江苏省", "长三角"],
    ["扬州", "江苏省", "长三角"],
    ["镇江", "江苏省", "长三角"],
    ["泰州", "江苏省", "长三角"],
    ["杭州", "浙江省", "长三角"],
    ["宁波", "浙江省", "长三角"],
    ["嘉兴", "浙江省", "长三角"],
    ["湖州", "浙江省", "长三角"],
    ["绍兴", "浙江省", "长三角"],
    ["金华", "浙江省", "长三角"],
    ["衢州", "浙江省", "长三角"],
    ["舟山", "浙江省", "长三角"],
    ["台州", "浙江省", "长三角"],
    ["合肥", "安徽省", "长三角"],
    ["芜湖", "安徽省", "长三角"],
    ["马鞍山", "安徽省", "长三角"],
    ["铜陵", "安徽省", "长三角"],
    ["安庆", "安徽省", "长三角"],
    ["滁州", "安徽省", "长三角"],
    ["池州", "安徽省", "长三角"],
    ["宣城", "安徽省", "长三角"],
    ["广州", "广东省", "粤港澳大湾区"],
    ["深圳", "广东省", "粤港澳大湾区"],
    ["珠海", "广东省", "粤港澳大湾区"],
    ["佛山", "广东省", "粤港澳大湾区"],
    ["惠州", "广东省", "粤港澳大湾区"],
    ["东莞", "广东省", "粤港澳大湾区"],
    ["中山", "广东省", "粤港澳大湾区"],
    ["江门", "广东省", "粤港澳大湾区"],
    ["肇庆", "广东省", "粤港澳大湾区"],
    ["香港", "香港特别行政区", "粤港澳大湾区"],
    ["澳门", "澳门特别行政区", "粤港澳大湾区"],
]

# 保存路径：output/china_urban_clusters.csv
file_path = os.path.join("output", "china_urban_clusters.csv")

# 写入文件
with open(file_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("✅ CSV 文件已生成到 output 文件夹")
print(f"📁 路径：{file_path}")

✅ CSV 文件已生成到 output 文件夹
📁 路径：output\china_urban_clusters.csv


在清洗过的“merged_data.csv”文件中找到对应的长三角和珠三角的城市，分别计算长三角和珠三角城市每年度的gap_to_gdp数据，并绘制折线图。
1. 读取城市群分类数据和城市经济数据
2. 筛选长三角和珠三角城市
3. 计算预算缺口(gap)和gap_to_gdp指标
4. 统计年度平均值和总体比值
5. 生成对比折线图
6. 保存核心结果（对比图+汇总统计）

- 提示词相关信息：https://www.doubao.com/thread/waaba78d48f1eda8c

In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

def setup_chinese_font():
    """设置中文字体支持，避免中文乱码"""
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False

def create_output_dir(output_path='output'):
    """创建输出文件夹，若已存在则跳过"""
    Path(output_path).mkdir(parents=True, exist_ok=True)
    return output_path

def load_and_check_data(urban_cluster_path, merged_data_path):
    """读取并验证数据完整性"""
    print("=" * 60)
    print("1. 读取数据文件")
    print("=" * 60)
    
    # 读取核心数据
    try:
        urban_clusters_df = pd.read_csv('output/china_urban_clusters.csv')
        merged_data_df = pd.read_csv('data_clean/merged_data.csv')
        print("✅ 数据文件读取成功")
    except FileNotFoundError as e:
        print(f"❌ 错误：找不到文件 - {e.filename}")
        raise
    
    # 打印数据基本信息
    print(f"\n📊 城市群数据 (china_urban_clusters.csv):")
    print(f"   数据形状: {urban_clusters_df.shape} | 列名: {urban_clusters_df.columns.tolist()}")
    
    print(f"\n📊 城市经济数据 (merged_data.csv):")
    print(f"   数据形状: {merged_data_df.shape} | 列名: {merged_data_df.columns.tolist()}")
    
    # 验证城市群分类列存在
    if 'group' not in urban_clusters_df.columns:
        raise ValueError("❌ 城市群数据缺少'group'列，请检查数据格式")
    
    print(f"\n🏙️  城市群分类情况:")
    print(urban_clusters_df['group'].value_counts())
    
    return urban_clusters_df, merged_data_df

def filter_city_clusters(urban_clusters_df, merged_data_df):
    """筛选长三角和珠三角（粤港澳大湾区）的有效城市"""
    print("\n" + "=" * 60)
    print("2. 筛选有效分析城市")
    print("=" * 60)
    
    # 提取城市群城市列表（注：数据中珠三角对应"粤港澳大湾区"）
    changjiang_all = urban_clusters_df[urban_clusters_df['group'] == '长三角']['city'].tolist()
    zhujiang_all = urban_clusters_df[urban_clusters_df['group'] == '粤港澳大湾区']['city'].tolist()
    
    # 提取经济数据中的城市列表
    merged_cities = set(merged_data_df['city'].unique())
    
    # 筛选两边都存在的有效城市
    changjiang_valid = [city for city in changjiang_all if city in merged_cities]
    zhujiang_valid = [city for city in zhujiang_all if city in merged_cities]
    
    # 打印筛选结果
    print(f"\n🔵 长三角城市群:")
    print(f"   总城市数: {len(changjiang_all)} | 有效城市数: {len(changjiang_valid)}")
    print(f"   有效城市: {changjiang_valid}")
    
    print(f"\n🔴 珠三角城市群 (粤港澳大湾区):")
    print(f"   总城市数: {len(zhujiang_all)} | 有效城市数: {len(zhujiang_valid)}")
    print(f"   有效城市: {zhujiang_valid}")
    
    # 验证是否有有效城市
    if not changjiang_valid:
        raise ValueError("❌ 长三角无有效城市数据，请检查数据匹配性")
    if not zhujiang_valid:
        raise ValueError("❌ 珠三角无有效城市数据，请检查数据匹配性")
    
    return changjiang_valid, zhujiang_valid

def calculate_core_indicators(merged_data_df, changjiang_cities, zhujiang_cities):
    """计算预算缺口(gap)和gap_to_gdp核心指标"""
    print("\n" + "=" * 60)
    print("3. 计算核心经济指标")
    print("=" * 60)
    
    # 创建数据副本避免修改原数据
    data_processed = merged_data_df.copy()
    
    # 计算关键指标
    # gap = 预算支出 - 预算收入（正数表示缺口，负数表示盈余）
    data_processed['gap'] = data_processed['expend'] - data_processed['income']
    # gap_to_gdp = 预算缺口 / GDP（反映缺口相对经济规模的比例）
    data_processed['gap_to_gdp'] = data_processed['gap'] / data_processed['gdp']
    
    # 筛选两大城市群的数据
    changjiang_data = data_processed[data_processed['city'].isin(changjiang_cities)]
    zhujiang_data = data_processed[data_processed['city'].isin(zhujiang_cities)]
    
    # 打印计算结果概览
    print(f"\n📈 数据处理结果:")
    print(f"   分析年份范围: {sorted(data_processed['year'].unique())}")
    print(f"   长三角数据记录数: {len(changjiang_data)}")
    print(f"   珠三角数据记录数: {len(zhujiang_data)}")
    
    # 验证计算结果合理性
    if changjiang_data['gap_to_gdp'].isna().sum() > 0:
        print(f"⚠️  长三角数据存在{changjiang_data['gap_to_gdp'].isna().sum()}个缺失值")
    if zhujiang_data['gap_to_gdp'].isna().sum() > 0:
        print(f"⚠️  珠三角数据存在{zhujiang_data['gap_to_gdp'].isna().sum()}个缺失值")
    
    return changjiang_data, zhujiang_data

def generate_yearly_statistics(changjiang_data, zhujiang_data):
    """生成年度统计数据（城市平均值+总体比值）"""
    print("\n" + "=" * 60)
    print("4. 生成年度统计数据")
    print("=" * 60)
    
    # 长三角年度统计（双维度：城市平均值 + 总体比值）
    changjiang_yearly = changjiang_data.groupby('year').agg({
        'gap': 'sum',          # 年度总预算缺口
        'gdp': 'sum',          # 年度总GDP
        'gap_to_gdp': 'mean',  # 城市平均gap_to_gdp（反映个体平均水平）
        'city': 'count'        # 年度有效城市数量
    }).rename(columns={'city': 'city_count'}).reset_index()
    # 总体gap_to_gdp（总缺口/总GDP，反映整体规模比例）
    changjiang_yearly['overall_gap_to_gdp'] = changjiang_yearly['gap'] / changjiang_yearly['gdp']
    
    # 珠三角年度统计（同维度）
    zhujiang_yearly = zhujiang_data.groupby('year').agg({
        'gap': 'sum',
        'gdp': 'sum',
        'gap_to_gdp': 'mean',
        'city': 'count'
    }).rename(columns={'city': 'city_count'}).reset_index()
    zhujiang_yearly['overall_gap_to_gdp'] = zhujiang_yearly['gap'] / zhujiang_yearly['gdp']
    
    # 打印前5年统计结果预览
    print(f"\n🔵 长三角年度统计（前5年）:")
    print(changjiang_yearly[['year', 'gap', 'gdp', 'gap_to_gdp', 'overall_gap_to_gdp']].head().round(4))
    
    print(f"\n🔴 珠三角年度统计（前5年）:")
    print(zhujiang_yearly[['year', 'gap', 'gdp', 'gap_to_gdp', 'overall_gap_to_gdp']].head().round(4))
    
    return changjiang_yearly, zhujiang_yearly

def create_comparison_chart(changjiang_yearly, zhujiang_yearly, output_dir):
    """创建长三角vs珠三角gap_to_gdp对比折线图"""
    print("\n" + "=" * 60)
    print("5. 生成可视化对比图表")
    print("=" * 60)
    
    # 设置图表样式（2个子图：城市平均值 + 总体比值）
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))
    color_changjiang = '#2E86AB'  # 长三角：深蓝色
    color_zhujiang = '#A23B72'    # 珠三角：深红色
    marker_size = 6
    line_width = 2.5
    
    # ---------------------- 子图1：城市平均gap_to_gdp ----------------------
    ax1.plot(changjiang_yearly['year'], changjiang_yearly['gap_to_gdp'],
             marker='o', linewidth=line_width, markersize=marker_size,
             color=color_changjiang, label='长三角 (城市平均值)',
             markerfacecolor='white', markeredgewidth=1.5)
    
    ax1.plot(zhujiang_yearly['year'], zhujiang_yearly['gap_to_gdp'],
             marker='s', linewidth=line_width, markersize=marker_size,
             color=color_zhujiang, label='珠三角 (城市平均值)',
             markerfacecolor='white', markeredgewidth=1.5)
    
    # 子图1样式配置
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_title('长三角 vs 珠三角城市群年度gap_to_gdp对比（城市平均值）',
                  fontsize=16, fontweight='bold', pad=20)
    ax1.set_xlabel('年份', fontsize=12)
    ax1.set_ylabel('gap_to_gdp（预算缺口/GDP）', fontsize=12)
    ax1.legend(fontsize=11, loc='upper left')
    ax1.set_xticks(range(min(changjiang_yearly['year']), max(changjiang_yearly['year']) + 1, 2))
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.2%}'))  # 百分比显示
    
    # ---------------------- 子图2：总体gap_to_gdp ----------------------
    ax2.plot(changjiang_yearly['year'], changjiang_yearly['overall_gap_to_gdp'],
             marker='o', linewidth=line_width, markersize=marker_size,
             color=color_changjiang, label='长三角 (总体比值)',
             markerfacecolor='white', markeredgewidth=1.5)
    
    ax2.plot(zhujiang_yearly['year'], zhujiang_yearly['overall_gap_to_gdp'],
             marker='s', linewidth=line_width, markersize=marker_size,
             color=color_zhujiang, label='珠三角 (总体比值)',
             markerfacecolor='white', markeredgewidth=1.5)
    
    # 子图2样式配置
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_title('长三角 vs 珠三角城市群年度gap_to_gdp对比（总体比值）',
                  fontsize=16, fontweight='bold', pad=20)
    ax2.set_xlabel('年份', fontsize=12)
    ax2.set_ylabel('gap_to_gdp（总预算缺口/总GDP）', fontsize=12)
    ax2.legend(fontsize=11, loc='upper left')
    ax2.set_xticks(range(min(changjiang_yearly['year']), max(changjiang_yearly['year']) + 1, 2))
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.2%}'))  # 百分比显示
    
    # 调整布局并保存
    plt.tight_layout()
    chart_path = os.path.join(output_dir, '04_gap_to_gdp_pearl and yangtze_river_delta.png')
    plt.savefig(chart_path, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    
    print(f"✅ 对比图表已保存至: {chart_path}")
    return chart_path

def create_summary_statistics(changjiang_yearly, zhujiang_yearly):
    """生成关键指标汇总统计表"""
    print("\n" + "=" * 60)
    print("6. 生成汇总统计分析")
    print("=" * 60)
    
    # 计算核心统计指标（2006-2024年完整周期）
    summary_data = {
        '指标': [
            '2006-2024年平均值',
            '最大值',
            '最小值',
            '最大值年份',
            '最小值年份',
            '2024年最新值'
        ],
        '长三角(城市平均)': [
            f"{changjiang_yearly['gap_to_gdp'].mean():.4f} ({changjiang_yearly['gap_to_gdp'].mean()*100:.2f}%)",
            f"{changjiang_yearly['gap_to_gdp'].max():.4f} ({changjiang_yearly['gap_to_gdp'].max()*100:.2f}%)",
            f"{changjiang_yearly['gap_to_gdp'].min():.4f} ({changjiang_yearly['gap_to_gdp'].min()*100:.2f}%)",
            f"{int(changjiang_yearly.loc[changjiang_yearly['gap_to_gdp'].idxmax(), 'year'])}",
            f"{int(changjiang_yearly.loc[changjiang_yearly['gap_to_gdp'].idxmin(), 'year'])}",
            f"{changjiang_yearly[changjiang_yearly['year']==2024]['gap_to_gdp'].iloc[0]:.4f} "
            f"({changjiang_yearly[changjiang_yearly['year']==2024]['gap_to_gdp'].iloc[0]*100:.2f}%)"
        ],
        '珠三角(城市平均)': [
            f"{zhujiang_yearly['gap_to_gdp'].mean():.4f} ({zhujiang_yearly['gap_to_gdp'].mean()*100:.2f}%)",
            f"{zhujiang_yearly['gap_to_gdp'].max():.4f} ({zhujiang_yearly['gap_to_gdp'].max()*100:.2f}%)",
            f"{zhujiang_yearly['gap_to_gdp'].min():.4f} ({zhujiang_yearly['gap_to_gdp'].min()*100:.2f}%)",
            f"{int(zhujiang_yearly.loc[zhujiang_yearly['gap_to_gdp'].idxmax(), 'year'])}",
            f"{int(zhujiang_yearly.loc[zhujiang_yearly['gap_to_gdp'].idxmin(), 'year'])}",
            f"{zhujiang_yearly[zhujiang_yearly['year']==2024]['gap_to_gdp'].iloc[0]:.4f} "
            f"({zhujiang_yearly[zhujiang_yearly['year']==2024]['gap_to_gdp'].iloc[0]*100:.2f}%)"
        ]
    }
    
    # 转换为DataFrame便于展示和保存
    summary_df = pd.DataFrame(summary_data)
    
    # 打印汇总结果
    print("\n📋 长三角 vs 珠三角 gap_to_gdp 汇总分析（2006-2024）:")
    print(summary_df.to_string(index=False))
    
    return summary_df

def save_core_results(summary_df, output_dir):
    """保存核心结果文件（仅保留对比图和汇总统计）"""
    print("\n" + "=" * 60)
    print("7. 保存核心结果文件")
    print("=" * 60)
    
    # 保存汇总统计CSV
    summary_path = os.path.join(output_dir, '04_gap_to_gdp total_pearl and yangtze_river_delta.csv')
    summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
    
    # 打印保存结果
    print(f"\n✅ 已保存核心文件:")
    print(f"   1. 04_gap_to_gdp_pearl and yangtze_river_delta.png（可视化图表）")
    print(f"   2. gap_to_gdp汇总统计.csv（关键指标汇总）")
    
    # 验证文件是否存在
    if os.path.exists(summary_path):
        print(f"\n📊 文件验证:")
        print(f"   汇总统计文件大小: {os.path.getsize(summary_path)/1024:.1f} KB")
        chart_path = os.path.join(output_dir, '04_gap_to_gdp_pearl and yangtze_river_delta.png')
        if os.path.exists(chart_path):
            print(f"   对比图表文件大小: {os.path.getsize(chart_path)/1024:.1f} KB")
        else:
            print(f"⚠️  警告：对比图表文件不存在，请检查生成流程")
    
    return summary_path

def main():
    """主函数：串联所有分析流程"""
    # 1. 初始化配置
    setup_chinese_font()
    output_dir = create_output_dir('output')  # 输出文件夹默认名为"output"
    
    # 2. 数据文件路径（需确保与代码同目录，或修改为实际路径）
    urban_cluster_path = 'china_urban_clusters.csv'
    merged_data_path = 'merged_data.csv'
    
    try:
        # 3. 数据加载与验证
        urban_clusters_df, merged_data_df = load_and_check_data(urban_cluster_path, merged_data_path)
        
        # 4. 有效城市筛选
        changjiang_cities, zhujiang_cities = filter_city_clusters(urban_clusters_df, merged_data_df)
        
        # 5. 核心指标计算
        changjiang_data, zhujiang_data = calculate_core_indicators(merged_data_df, changjiang_cities, zhujiang_cities)
        
        # 6. 年度统计生成
        changjiang_yearly, zhujiang_yearly = generate_yearly_statistics(changjiang_data, zhujiang_data)
        
        # 7. 可视化图表生成
        create_comparison_chart(changjiang_yearly, zhujiang_yearly, output_dir)
        
        # 8. 汇总统计生成
        summary_df = create_summary_statistics(changjiang_yearly, zhujiang_yearly)
        
        # 9. 核心结果保存
        save_core_results(summary_df, output_dir)
        
        # 10. 分析完成
        print("\n" + "=" * 80)
        print("🎉 长三角和珠三角gap_to_gdp分析全部完成！")
        print(f"📁 所有核心结果已保存至: {os.path.abspath(output_dir)}")
        print("=" * 80)
        
    except Exception as e:
        print(f"\n❌ 分析过程出错: {str(e)}")
        print("请检查数据文件格式或运行环境后重试")

# 程序入口
if __name__ == "__main__":
    main()

1. 读取数据文件
✅ 数据文件读取成功

📊 城市群数据 (china_urban_clusters.csv):
   数据形状: (38, 3) | 列名: ['city', 'province', 'group']

📊 城市经济数据 (merged_data.csv):
   数据形状: (684, 6) | 列名: ['city', 'year', 'income', 'expend', 'gdp', 'deposit']

🏙️  城市群分类情况:
group
长三角       27
粤港澳大湾区    11
Name: count, dtype: int64

2. 筛选有效分析城市

🔵 长三角城市群:
   总城市数: 27 | 有效城市数: 5
   有效城市: ['上海', '南京', '杭州', '宁波', '合肥']

🔴 珠三角城市群 (粤港澳大湾区):
   总城市数: 11 | 有效城市数: 2
   有效城市: ['广州', '深圳']

3. 计算核心经济指标

📈 数据处理结果:
   分析年份范围: [np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   长三角数据记录数: 95
   珠三角数据记录数: 38

4. 生成年度统计数据

🔵 长三角年度统计（前5年）:
   year     gap       gdp  gap_to_gdp  overall_gap_to_gdp
0  2006  268.42  20988.89      0.0105              0.0128
1  2007  103.73  25333.31      0.0018  